# 04 - Final Evaluation (Qwen3.5-4B V2 Fine-tuned)

Load best_mae_checkpoint, evaluate on 200 test items (seed=42), draw charts,
save results JSON. Run AFTER 03_train_v2 finishes.

In [ ]:
# Cell 1 - Imports + constants
import os, sys, json, torch
sys.path.insert(0, os.path.abspath("."))

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from utils.items_vn import load_items, DATASET_NAME
from utils.evaluator_vn import VnTester
from utils.training_utils import get_bnb_config, MAX_SEQ_LENGTH, MAX_NEW_TOKENS

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
BEST_CKPT    = "outputs/qwen_v2/best_mae_checkpoint"
HUB_MODEL_ID = "SeanSunny/qwen3.5-4b-vn-pricer-v2"
RESULTS_PATH = "results/v2_results.json"
EVAL_SIZE    = 200

In [ ]:
# Cell 2 - Load base + adapter. torch_dtype=bfloat16 BAT BUOC.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=get_bnb_config(),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, BEST_CKPT)
model.eval()
print(f"Loaded adapter from: {BEST_CKPT}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 3 - Test items (seed=42, 200 items)
import random; random.seed(42)
all_test = load_items("test")
test_items = random.sample(all_test, min(EVAL_SIZE, len(all_test)))
print(f"Test items: {len(test_items)}")
print(f"Price range: {min(i.price for i in test_items):.0f}K - {max(i.price for i in test_items):.0f}K VND")

In [ ]:
# Cell 4 - Predictor function
import re
def qwen_v2_predict(item) -> float:
    """Returns predicted price in K VND."""
    inputs = tokenizer(item.prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    text = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    m = re.search(r"\d+\.?\d*", text)
    return float(m.group()) if m else 0.0

In [ ]:
# Cell 5 - Evaluate + charts (scatter + error trend)
tester = VnTester(
    predictor=qwen_v2_predict,
    data=test_items,
    title="Qwen3.5-4B V2 Fine-tuned",
    size=EVAL_SIZE,
    workers=1,
)
tester.run()

In [ ]:
# Cell 6 - Save results JSON
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from utils.evaluator_vn import _rmsle

mae_k = float(np.mean(tester.errors))
mse   = float(mean_squared_error(tester.truths, tester.guesses))
r2    = float(r2_score(tester.truths, tester.guesses))
rmsle = _rmsle(tester.truths, tester.guesses)

results = {
    "model": HUB_MODEL_ID,
    "checkpoint": BEST_CKPT,
    "eval_size": EVAL_SIZE,
    "mae_k_vnd": round(mae_k, 2),
    "mae_vnd": round(mae_k * 1000, 0),
    "rmsle": round(rmsle, 4),
    "mse": round(mse, 2),
    "r2": round(r2, 4),
}
os.makedirs("results", exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(json.dumps(results, indent=2))